In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import joblib
import os
from meridian.analysis import optimizer, summarizer, summarizer_cm, GCPClient, ClientConfig

In [ ]:
# Cargar 'model.pkl'
model = "5 - Models_2025_12_geo.pkl"
mmm = joblib.load(model)

In [ ]:
# Definir variable de entorno con la ruta al archivo de credenciales de Google Cloud.
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/path/to/your/service_account_key.json"

# Crear cliente de GCP
gcp_client = GCPClient()

In [ ]:
# Crear configuración del cliente
client_config = ClientConfig(config_path="/path/to/your/client_config.yaml")

In [ ]:
# Crear el optimizador desde el modelo entrenado
budget_optimizer = optimizer.BudgetOptimizer(mmm, client_config)

# Optimización - el método detecta automáticamente si es geo o nacional
results = budget_optimizer.optimize(
    budget=1000000,
    fixed_budget=True,
    spend_constraint_lower=0.5,
    spend_constraint_upper=0.5,
    selected_geos=["Auto"],
)

results.output_optimization_summary(
    filename="optimization_output.html",
    filepath="./reports/",
)

# Guardar en GCS
gcp_client.upload_file_to_gcs(
    bucket_name="your-gcs-bucket-name",
    prefix="Reports",
    filepath="./reports/",
    filename="optimization_output.html",
)

In [ ]:
mmm_summarizer = summarizer.Summarizer(mmm, client_config)
start_date_ = "2025-12-29"
end_date_ = "2026-04-06"

mmm_summarizer.output_model_results_summary(
    filename="mmm_report.html",
    filepath="./reports/",
    start_date=start_date_,
    end_date=end_date_,
    selected_geos=["Auto"],
)

# Guardar en GCS
gcp_client.upload_file_to_gcs(
    bucket_name="your-gcs-bucket-name",
    prefix="Reports",
    filepath="./reports/",
    filename="mmm_report.html",
)

In [ ]:
mmm_summarizer_cm = summarizer_cm.Summarizer(mmm, client_config)
start_date_ = "2024-12-30"
end_date_ = "2025-12-22"
start_date_cm_ = "2024-01-01"
end_date_cm_ = "2024-12-23"

mmm_summarizer_cm.output_comparison_metrics_summary(
    filename="mmm_report_comparison_metrics.html",
    filepath="./reports/",
    start_date=start_date_,
    end_date=end_date_,
    start_date_cm=start_date_cm_,
    end_date_cm=end_date_cm_,
    filepath_cm="./reports/cm_tables/",
    selected_geos=["Auto"],
)

# Guardar en GCS
gcp_client.upload_file_to_gcs(
    bucket_name="your-gcs-bucket-name",
    prefix="Reports",
    filepath="./reports/",
    filename="mmm_report_comparison_metrics.html",
)

# Cargar en BigQuery
for file in os.listdir("./reports/cm_tables/"):
  if file.endswith(".parquet"):
    parquet_path = os.path.join("./reports/cm_tables/", file)
    gcp_client.load_parquet_to_bq(
        parquet_path=parquet_path,
        table_id="your-project-id.your_dataset.your_table",
    )